In [0]:
%pip install rasterio numpy>=2
%restart_python

In [0]:
import numpy as np
import os
import rasterio

from datetime import datetime, UTC

import pyspark.sql.types as ST
from pyspark.sql.datasource import DataSource, DataSourceReader, InputPartition
import pyspark.sql.functions as F

In [0]:
class BandPartition(InputPartition):
  def __init__(self, path: str, band: int):
    self.path = path
    self.band = band

class BandBlockgroupPartition(InputPartition):
  def __init__(self, path: str, blockgroup: int, band: int):
    self.path = path
    self.blockgroup = blockgroup
    self.band = band

In [0]:
class HLSVIDataSourceReader(DataSourceReader):
  def __init__(self, schema, options):
    self.schema: ST.StructType = schema
    self.options = options

  @staticmethod
  def string_to_datetime(timestamp: str) -> datetime:
    try:
      return datetime.fromisoformat(timestamp)
    except ValueError as e:
      raise Exception(f"Invalid timestamp format for {timestamp}. Expected format: YYYY-MM-DDTHH:MM:SS[.mmmmmm[+|-]hh:mm].\n{e}")
  
  @property
  def input_path(self):
    path: str = self.options.get("path")
    if not path:
      raise ValueError("The 'path' option is required.")
    return path
  
  @property
  def input_files(self):
    files = []
    if not os.path.exists(self.input_path):
      raise Exception(f"Input path '{self.input_path}' does not exist.")
    if os.path.isfile(self.input_path):
      if self.input_path.endswith(".tif") and not "mask" in self.input_path:
        files.append(self.input_path)
      return files
    for root, _, filenames in os.walk(self.input_path):
      for filename in filenames:
        if filename.endswith(".tif") and not "mask" in filename:
          files.append(os.path.join(root, filename))
    return files
  
  @property
  def blockgroup_size(self):
    size: int = int(self.options.get("blockgroup_size", "-1"))
    return size
  
  def partitions(self):
    import rasterio
    parts = []
    for path in self.input_files:
      with rasterio.open(path) as src:
        if self.blockgroup_size == -1:
          parts += [BandPartition(path, b + 1) for b in range(src.count)]
        else:
          parts += [BandBlockgroupPartition(path, g, b + 1) for b in range(src.count) for g in range(src.height // self.blockgroup_size)]
    return parts

  def read(self, partition):
    import rasterio
    with rasterio.open(partition.path) as r:
      metadata_tags = r.tags()
      
      sensing_time_str = metadata_tags["SENSING_TIME"].split(";")[-1].replace(" ", "")
      sensing_time = self.string_to_datetime(sensing_time_str)
      del(metadata_tags["SENSING_TIME"])

      variable = metadata_tags["long_name"]
      del(metadata_tags["long_name"])
      
      image_height, image_width = r.shape
      cols, rows = np.meshgrid(np.arange(image_width), np.arange(image_height))
      xs, ys = rasterio.transform.xy(r.transform, rows, cols) # , offset="ul"
      if isinstance(partition, BandPartition):
        values = r.read(partition.band)
      else:
        values = r.read(partition.band, window=rasterio.windows.Window(0, self.blockgroup_size * partition.blockgroup, image_width, self.blockgroup_size))
      nodata_value = r.nodatavals[partition.band - 1]
      srid = r.crs.to_epsg()
      rows = zip(xs, ys, values.astype(np.int16).flatten())
      for rw in rows:
        if rw[2] != nodata_value:
          yield (partition.path, metadata_tags, srid, partition.band, sensing_time, variable, nodata_value, *rw)

In [0]:
class HLSVIDataSource(DataSource):

    @classmethod
    def name(cls) -> str:
        """
        Get the name of the data source.

        Returns:
            str: The name of the data source.
        """
        return "hlsvi"
    
    def schema(self) -> ST.StructType:
        """
        Define the schema for the output data.

        Returns:
            StructType: The schema including fields for the variable identifier, band index,
            metadata, coordinates, and values.
        """
        return ST.StructType([
            ST.StructField("path", ST.StringType(), True),
            ST.StructField("metadata", ST.MapType(ST.StringType(), ST.StringType()), True),
            ST.StructField("srid", ST.IntegerType(), True),
            ST.StructField("band", ST.IntegerType(), True),
            ST.StructField("sensing_time", ST.TimestampType(), True),
            ST.StructField("variable", ST.StringType(), True),
            ST.StructField("nodata", ST.IntegerType(), True),
            ST.StructField("x", ST.DoubleType(), True),
            ST.StructField("y", ST.DoubleType(), True),
            ST.StructField("m", ST.DoubleType(), True),
        ])

    def reader(self, schema: ST.StructType):
        return HLSVIDataSourceReader(schema, self.options)

In [0]:
spark.dataSource.register(HLSVIDataSource)

In [0]:
catalog = "stuart"
schema = "lv"

raw_df = (
  spark.read
  .format("hlsvi")
  .option("blockgroup_size", str(3600 // 10))
  .load(f"/Volumes/{catalog}/{schema}/geotiffs")
  )

In [0]:
raw_tref = f"{catalog}.{schema}.raw_raster"

In [0]:
raw_df.write.mode("overwrite").saveAsTable(raw_tref)

In [0]:
display(spark.table(raw_tref))

In [0]:
print(f"{spark.table(raw_tref).count()=:,}")